# Nested Cross-Validation for Medical ML## Unbiased Model Selection and EvaluationThis notebook demonstrates nested cross-validation, a critical technique for:- Hyperparameter tuning without overfitting- Unbiased performance estimation- Proper model selection in medical applications

In [ ]:
# Import required librariesimport numpy as npimport pandas as pdimport matplotlib.pyplot as pltimport seaborn as snsfrom sklearn.datasets import make_classificationfrom sklearn.ensemble import RandomForestClassifierfrom sklearn.svm import SVCfrom sklearn.linear_model import LogisticRegressionfrom sklearn.model_selection import GridSearchCV, cross_val_scorefrom sklearn.metrics import accuracy_score, roc_auc_score, classification_reportimport warningswarnings.filterwarnings('ignore')# Set styleplt.style.use('seaborn-v0_8-darkgrid')sns.set_palette(['#870052', '#FF876F', '#4CAF50', '#2196F3'])

## 1. The Problem with Simple Cross-ValidationWhen we use the same CV to both select hyperparameters AND evaluate performance, we get **optimistically biased** results.

In [ ]:
# Generate synthetic medical dataX, y = make_classification(    n_samples=500,    n_features=30,    n_informative=10,    n_redundant=10,    n_clusters_per_class=2,    weights=[0.7, 0.3],  # Imbalanced like many medical datasets    random_state=42)print(f"Dataset shape: {X.shape}")print(f"Class distribution: {np.bincount(y) / len(y)}")

### Wrong Way: Using same CV for tuning and evaluation

In [ ]:
# WRONG: This gives optimistically biased resultsfrom sklearn.model_selection import StratifiedKFold# Define model and parametersrf_model = RandomForestClassifier(random_state=42)param_grid = {    'n_estimators': [50, 100, 200],    'max_depth': [5, 10, None],    'min_samples_split': [2, 5, 10]}# Single CV for both tuning and evaluation (WRONG!)cv = StratifiedKFoldMedical(n_splits=5, shuffle=True, random_state=42)grid_search = GridSearchCV(rf_model, param_grid, cv=cv, scoring='roc_auc')grid_search.fit(X, y)print("❌ BIASED approach (same CV for tuning & evaluation):")print(f"Best parameters: {grid_search.best_params_}")print(f"Best CV score: {grid_search.best_score_:.3f}")print("\n⚠️ This score is optimistically biased!")

## 2. Nested Cross-Validation: The Correct ApproachNested CV uses:- **Outer loop**: Unbiased performance estimation- **Inner loop**: Hyperparameter tuning

In [ ]:
# Import our trustcv implementationimport syssys.path.append('..')from trustcv.splitters.iid import NestedCV# Or implement it manuallydef nested_cross_validation(X, y, model, param_grid,                           outer_cv, inner_cv, scoring='roc_auc'):    """    Perform nested cross-validation    """    outer_scores = []    best_params_list = []        for fold_idx, (train_idx, test_idx) in enumerate(outer_cv.split(X, y)):        # Split data for outer fold        X_train, X_test = X[train_idx], X[test_idx]        y_train, y_test = y[train_idx], y[test_idx]                # Inner CV for hyperparameter tuning        grid_search = GridSearchCV(            model, param_grid, cv=inner_cv,             scoring=scoring, n_jobs=-1        )        grid_search.fit(X_train, y_train)                # Evaluate on outer test set        if scoring == 'roc_auc':            y_pred_proba = grid_search.predict_proba(X_test)[:, 1]            score = roc_auc_score(y_test, y_pred_proba)        else:            y_pred = grid_search.predict(X_test)            score = accuracy_score(y_test, y_pred)                outer_scores.append(score)        best_params_list.append(grid_search.best_params_)                print(f"Outer fold {fold_idx + 1}: Score = {score:.3f}")        print(f"  Best params: {grid_search.best_params_}")        return outer_scores, best_params_list

In [ ]:
# Perform nested CVouter_cv = StratifiedKFoldMedical(n_splits=5, shuffle=True, random_state=42)inner_cv = StratifiedKFoldMedical(n_splits=3, shuffle=True, random_state=42)print("✅ UNBIASED approach (nested CV):")print("="*50)outer_scores, best_params = nested_cross_validation(    X, y,     RandomForestClassifier(random_state=42),    param_grid,    outer_cv,    inner_cv)print("\n" + "="*50)print(f"Unbiased performance: {np.mean(outer_scores):.3f} ± {np.std(outer_scores):.3f}")print("\n📊 Note: This is typically lower than the biased estimate!")

## 3. Comparing Multiple Models with Nested CV

In [ ]:
# Define multiple models and their parameter gridsmodels = {    'Random Forest': {        'model': RandomForestClassifier(random_state=42),        'params': {            'n_estimators': [50, 100],            'max_depth': [5, 10, None]        }    },    'SVM': {        'model': SVC(probability=True, random_state=42),        'params': {            'C': [0.1, 1, 10],            'kernel': ['rbf', 'linear']        }    },    'Logistic Regression': {        'model': LogisticRegression(max_iter=1000, random_state=42),        'params': {            'C': [0.01, 0.1, 1, 10],            'penalty': ['l2']        }    }}# Compare modelsresults = {}for name, config in models.items():    print(f"\nEvaluating {name}...")    print("-" * 30)        scores, params = nested_cross_validation(        X, y,        config['model'],        config['params'],        outer_cv,        inner_cv    )        results[name] = {        'scores': scores,        'mean': np.mean(scores),        'std': np.std(scores),        'params': params    }

In [ ]:
# Visualize resultsfig, axes = plt.subplots(1, 2, figsize=(14, 5))# Box plot of scoresax1 = axes[0]scores_data = [results[name]['scores'] for name in models.keys()]bp = ax1.boxplot(scores_data, labels=models.keys(), patch_artist=True)for patch, color in zip(bp['boxes'], ['#870052', '#FF876F', '#4CAF50']):    patch.set_facecolor(color)    patch.set_alpha(0.7)ax1.set_ylabel('ROC-AUC Score')ax1.set_title('Model Comparison with Nested CV')ax1.grid(True, alpha=0.3)# Bar plot of mean scores with error barsax2 = axes[1]names = list(models.keys())means = [results[name]['mean'] for name in names]stds = [results[name]['std'] for name in names]colors = ['#870052', '#FF876F', '#4CAF50']bars = ax2.bar(names, means, yerr=stds, capsize=10,                color=colors, alpha=0.7, edgecolor='black', linewidth=2)ax2.set_ylabel('Mean ROC-AUC Score')ax2.set_title('Mean Performance ± Std Dev')ax2.set_ylim([0, 1])ax2.grid(True, alpha=0.3, axis='y')# Add value labelsfor bar, mean, std in zip(bars, means, stds):    height = bar.get_height()    ax2.text(bar.get_x() + bar.get_width()/2., height + std + 0.01,             f'{mean:.3f}', ha='center', va='bottom')plt.tight_layout()plt.show()# Print summaryprint("\n" + "="*50)print("SUMMARY - Unbiased Model Comparison")print("="*50)for name in models.keys():    print(f"{name:20s}: {results[name]['mean']:.3f} ± {results[name]['std']:.3f}")    best_model = max(results.keys(), key=lambda x: results[x]['mean'])print(f"\n🏆 Best model: {best_model}")

## 4. Nested CV for Grouped Medical DataWhen dealing with patient-grouped data, both loops must respect grouping!

In [ ]:
# Generate grouped medical data (multiple records per patient)n_patients = 100records_per_patient = 5n_samples = n_patients * records_per_patient# Create data with patient structureX_grouped, y_grouped = make_classification(    n_samples=n_samples,    n_features=20,    n_informative=10,    n_clusters_per_class=3,    weights=[0.6, 0.4],    random_state=42)# Create patient IDspatient_ids = np.repeat(np.arange(n_patients), records_per_patient)print(f"Total samples: {n_samples}")print(f"Unique patients: {n_patients}")print(f"Records per patient: {records_per_patient}")

In [ ]:
from sklearn.model_selection import GroupKFolddef nested_grouped_cv(X, y, groups, model, param_grid):    """    Nested CV that respects patient grouping    """    outer_cv = GroupKFoldMedical(n_splits=5)    inner_cv = GroupKFoldMedical(n_splits=3)        outer_scores = []        for fold_idx, (train_idx, test_idx) in enumerate(outer_cv.split(X, y, groups)):        X_train, X_test = X[train_idx], X[test_idx]        y_train, y_test = y[train_idx], y[test_idx]        groups_train = groups[train_idx]                # Inner CV with grouping        grid_search = GridSearchCV(            model, param_grid,             cv=inner_cv,            scoring='roc_auc'        )        grid_search.fit(X_train, y_train, groups=groups_train)                # Evaluate        y_pred_proba = grid_search.predict_proba(X_test)[:, 1]        score = roc_auc_score(y_test, y_pred_proba)        outer_scores.append(score)                # Check patient separation        train_patients = set(groups[train_idx])        test_patients = set(groups[test_idx])        assert len(train_patients & test_patients) == 0, "Data leakage detected!"                print(f"Fold {fold_idx + 1}: {score:.3f} "              f"(train: {len(train_patients)} patients, "              f"test: {len(test_patients)} patients)")        return outer_scores# Run nested grouped CVprint("Nested CV with Patient Grouping")print("="*50)grouped_scores = nested_grouped_cv(    X_grouped, y_grouped, patient_ids,    RandomForestClassifier(random_state=42),    {'n_estimators': [50, 100], 'max_depth': [5, 10]})print(f"\nGrouped nested CV score: {np.mean(grouped_scores):.3f} ± {np.std(grouped_scores):.3f}")print("✅ No patient appears in both train and test!")

## 5. Nested CV for Time Series Medical Data

In [ ]:
# Generate temporal medical datan_timepoints = 500n_features = 15# Create time series with trend and seasonalitynp.random.seed(42)time = np.arange(n_timepoints)trend = time * 0.01seasonal = 10 * np.sin(2 * np.pi * time / 50)noise = np.random.randn(n_timepoints) * 5# Create features with temporal patternsX_temporal = np.zeros((n_timepoints, n_features))for i in range(n_features):    X_temporal[:, i] = trend + seasonal * (i % 3) + noise + np.random.randn(n_timepoints) * 2# Create target with temporal dependencyy_temporal = (X_temporal[:, 0] + X_temporal[:, 1] > np.median(X_temporal[:, 0] + X_temporal[:, 1])).astype(int)print(f"Temporal data shape: {X_temporal.shape}")print(f"Time range: {n_timepoints} time points")

In [ ]:
from sklearn.model_selection import TimeSeriesSplitdef nested_temporal_cv(X, y, model, param_grid):    """    Nested CV for time series data    """    outer_cv = TimeSeriesSplit(n_splits=5)    inner_cv = TimeSeriesSplit(n_splits=3)        outer_scores = []    train_sizes = []        for fold_idx, (train_idx, test_idx) in enumerate(outer_cv.split(X)):        X_train, X_test = X[train_idx], X[test_idx]        y_train, y_test = y[train_idx], y[test_idx]                # Inner CV for hyperparameter tuning        grid_search = GridSearchCV(            model, param_grid,             cv=inner_cv,            scoring='accuracy'  # Using accuracy for binary classification        )        grid_search.fit(X_train, y_train)                # Evaluate        y_pred = grid_search.predict(X_test)        score = accuracy_score(y_test, y_pred)        outer_scores.append(score)        train_sizes.append(len(train_idx))                print(f"Fold {fold_idx + 1}: {score:.3f} "              f"(train: {train_idx[0]}-{train_idx[-1]}, "              f"test: {test_idx[0]}-{test_idx[-1]})")        return outer_scores, train_sizes# Run nested temporal CVprint("Nested CV for Time Series")print("="*50)temporal_scores, train_sizes = nested_temporal_cv(    X_temporal, y_temporal,    RandomForestClassifier(random_state=42),    {'n_estimators': [30, 50], 'max_depth': [3, 5]})print(f"\nTemporal nested CV score: {np.mean(temporal_scores):.3f} ± {np.std(temporal_scores):.3f}")print("✅ Training data always precedes test data!")

In [ ]:
# Visualize temporal CV splits and performancefig, axes = plt.subplots(2, 1, figsize=(14, 8))# Plot 1: Learning curveax1 = axes[0]ax1.plot(train_sizes, temporal_scores, 'o-', color='#870052', linewidth=2, markersize=8)ax1.fill_between(train_sizes,                   [s - 0.05 for s in temporal_scores],                  [s + 0.05 for s in temporal_scores],                  alpha=0.3, color='#870052')ax1.set_xlabel('Training Set Size')ax1.set_ylabel('Accuracy')ax1.set_title('Learning Curve: Performance vs Training Size')ax1.grid(True, alpha=0.3)# Plot 2: Temporal splits visualizationax2 = axes[1]outer_cv = TimeSeriesSplit(n_splits=5)for fold_idx, (train_idx, test_idx) in enumerate(outer_cv.split(X_temporal)):    # Plot train    ax2.barh(fold_idx, len(train_idx), left=train_idx[0],              color='#870052', alpha=0.7, height=0.8)    # Plot test    ax2.barh(fold_idx, len(test_idx), left=test_idx[0],              color='#FF876F', alpha=0.7, height=0.8)ax2.set_xlabel('Time Index')ax2.set_ylabel('CV Fold')ax2.set_title('Temporal Cross-Validation Splits')ax2.legend(['Training', 'Test'], loc='upper left')ax2.grid(True, alpha=0.3, axis='x')plt.tight_layout()plt.show()

## 6. Best Practices and Recommendations### When to Use Nested CV:1. **Always** when reporting final model performance2. When comparing multiple models fairly3. For publication-ready results4. In regulatory submissions (FDA, CE marking)### Common Pitfalls to Avoid:1. ❌ Using test set for ANY decisions2. ❌ Selecting features using all data3. ❌ Not respecting data structure (groups, time)4. ❌ Information leakage between folds### Computational Considerations:- Nested CV is expensive: O(k_outer × k_inner × training_time)- Use parallel processing when possible- Consider fewer hyperparameter combinations- Start with smaller parameter grids

In [ ]:
# Final comparison: Biased vs Unbiased estimatesprint("="*60)print("FINAL COMPARISON: Why Nested CV Matters")print("="*60)# Biased estimate (wrong way)simple_cv = StratifiedKFoldMedical(n_splits=5, shuffle=True, random_state=42)grid = GridSearchCV(RandomForestClassifier(random_state=42), param_grid, cv=simple_cv, scoring='roc_auc')grid.fit(X, y)biased_score = grid.best_score_# Unbiased estimate (correct way)unbiased_score = np.mean(outer_scores)print(f"\n❌ Biased estimate (simple CV):   {biased_score:.3f}")print(f"✅ Unbiased estimate (nested CV): {unbiased_score:.3f}")print(f"\n📊 Optimism bias: {(biased_score - unbiased_score):.3f}")print(f"   This {(biased_score - unbiased_score)*100:.1f}% overestimation could lead to:")print("   - Failed clinical trials")print("   - Poor real-world performance")print("   - Regulatory rejection")print("\n💡 Always use nested CV for honest performance estimates!")